# Tornado dataset: records, footprints, and county context

Explore the **2010–2025** collection of tornado records, EF ratings, and damage
footprints. This notebook only reads local files. Run
`uv run python download_data.py --verify-downloads` to collect and verify data.
See [README.md](../README.md) and [DATASET.md](../docs/DATASET.md).

| Source | Unit / purpose |
|---|---|
| SPC | Historical tornado tracks and candidate EF labels |
| NCEI Storm Events | County/event details, fatalities, locations, and narratives |
| NOAA Event Footprint Catalog | DAT/Storm Events damage footprints; regions are not unique tornadoes |
| Census Population Estimates | Annual county population and housing units |
| Census Cartographic Boundaries | Fixed 2020 simplified county map |

These sources count different units; their row totals cannot be added to count
tornadoes. Missing survey records do not mean no tornado occurred. Unknown EF
ratings are retained and must not be converted to EF0.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'download_data.py').is_file() and (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the tornado-classification repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from dataset_inspection import (load_json, local_path, csv_preview,
                                inventory, reference_check, footprint_preview)

DATA = ROOT / 'data'  # Change only if you used download_data.py --data-dir.
START_YEAR, END_YEAR = 2010, 2025
SAMPLE_ROWS = 10
NCEI_YEAR = 2025
pd.set_option('display.max_columns', 16)
pd.set_option('display.max_colwidth', 80)
print(f'Inspecting {DATA}; period {START_YEAR}–{END_YEAR}')
print('All cells are read-only. No downloads are triggered.')


## Download status and source inventory

The NOAA download manifest covers SPC, NCEI, and EFC; a separate Census manifest
covers the two context sources. Check the verification scope as well as its status. The independent verification
report checks saved files, filtering, and annual footprints. Successful retrieval
does not establish complete historical survey coverage. Inventory includes caches;
use the manifest to identify the active selection.


In [ ]:
manifest = load_json(DATA / f'download_manifest_{START_YEAR}_{END_YEAR}.json', {})
quality = load_json(DATA / f'quality_summary_{START_YEAR}_{END_YEAR}.json', {})
verification = load_json(DATA / f'download_verification_{START_YEAR}_{END_YEAR}.json', {})
census_manifest = load_json(DATA / f'census_manifest_{START_YEAR}_{END_YEAR}.json', {})
status = [
    {'scope': 'SPC / NCEI / EFC',
     'status': 'available' if manifest.get('requested_archives_available') else 'incomplete' if manifest else 'not downloaded'},
    {'scope': 'Census estimates / county map', 'status': census_manifest.get('status', 'not downloaded')},
    {'scope': 'Verification: ' + verification.get('scope', 'unknown'), 'status': verification.get('status', 'not run')},
]
display(pd.DataFrame(status))
if verification.get('manifest'):
    display(pd.DataFrame([reference_check(DATA, verification['manifest'])]))
if verification.get('census_manifest'):
    display(pd.DataFrame([reference_check(DATA, verification['census_manifest'])]))
files = pd.DataFrame(inventory(DATA))
if not files.empty:
    display(files)
else:
    print('No dataset files yet. Run the download script separately when ready.')


## Start with the consolidated analysis tables (v2.0.0)

Run `uv run python scripts/build_analysis.py` after collecting or refreshing sources.
Start with `tornadoes.parquet`; the other tables provide optional NCEI details,
footprints, and Census context. The table inventory below lists all seven main tables
plus the annual summary. Source hashes are checked before loading. These files
preserve separate record types and do not perform cross-source event matching.


In [ ]:
import hashlib
analysis_dir = DATA / 'analysis'
analysis_manifest = load_json(analysis_dir / 'manifest.json', {})
if not analysis_manifest:
    print('Analysis tables are not built yet. Run scripts/build_analysis.py.')
else:
    stale = []
    for entry in analysis_manifest['inputs']:
        path = local_path(DATA, entry['path'])
        if not path.is_file():
            stale.append(entry['path'])
        else:
            with path.open('rb') as handle:
                if hashlib.file_digest(handle, 'sha256').hexdigest() != entry['sha256']:
                    stale.append(entry['path'])
    for entry in analysis_manifest['files']:
        path = local_path(analysis_dir, entry['path'])
        if not path.is_file():
            stale.append(entry['path'])
        else:
            with path.open('rb') as handle:
                if hashlib.file_digest(handle, 'sha256').hexdigest() != entry['sha256']:
                    stale.append(entry['path'])
    if stale:
        print('Analysis is stale or incomplete; rebuild before using it:', stale[:5])
    else:
        display(pd.DataFrame([{k:entry[k] for k in ['path','rows','bytes']}
                              for entry in analysis_manifest['files']]))
        tornadoes = pd.read_parquet(analysis_dir / 'tornadoes.parquet')
        county_analysis = pd.read_parquet(analysis_dir / 'county_context.parquet')
        annual_summary = pd.read_csv(analysis_dir / 'annual_summary.csv')
        print(f'{len(tornadoes):,} tracks; {tornadoes.ef_rating.isna().sum():,} unknown EF ratings')
        display(tornadoes.head(SAMPLE_ROWS))
        display(county_analysis.head(SAMPLE_ROWS))
        display(annual_summary)
        if not stale and 'storm_events.parquet' in {e['path'] for e in analysis_manifest['files']}:
            import geopandas as gpd
            events = pd.read_parquet(analysis_dir / 'storm_events.parquet')
            footprints = gpd.read_parquet(analysis_dir / 'tornado_footprints.parquet',
                                           columns=['footprint_id','source','ef_rating','width_is_placeholder','geometry'])
            boundaries = gpd.read_parquet(analysis_dir / 'county_boundaries.parquet')
            display(events[['event_id','source_ef_rating','ef_rating','source_file']].head(SAMPLE_ROWS))
            display(footprints.head(SAMPLE_ROWS))
            print('Footprint CRS:', footprints.crs.to_string(), '| County map features:', len(boundaries))


## Optional SPC-centered linkage

Run `uv run python scripts/build_crosswalk.py` to build the separate local linkage layer. It preserves all seven source-oriented tables and all SPC rows. Accepted links are unique close matches in time and geometry; borderline, ambiguous, and unmatched source records remain visible. These are rule-based research links, not manually verified identities or part of the frozen v2.0.0 host release.

See [linkage methods](../docs/LINKAGE.md) and the [audit](../reports/linkage/audit.md). County totals describe linked county-years, not people/buildings struck. The linked table preserves post-event SPC fields and is not a selected predictor matrix.

In [ ]:
linkage_dir = DATA / 'linkage'
linkage_manifest = load_json(linkage_dir / 'manifest.json', {})
if not linkage_manifest:
    print('Linkage is not built yet. Run scripts/build_crosswalk.py.')
else:
    stale = [entry['path'] for entry in linkage_manifest['inputs']
             if reference_check(DATA, entry)['status'] != 'matches']
    if stale:
        raise ValueError(f'Linkage inputs changed; rebuild scripts/build_crosswalk.py: {stale}')
    for entry in linkage_manifest['files']:
        check = dict(entry, path='linkage/' + entry['path'])
        if reference_check(DATA, check)['status'] != 'matches':
            raise ValueError(f"Linkage file changed: {entry['path']}")
    display(pd.DataFrame(linkage_manifest['summary']['sources']))
    linked_tornadoes = pd.read_parquet(linkage_dir / 'tornadoes_linked.parquet')
    crosswalk = pd.read_parquet(linkage_dir / 'source_crosswalk.parquet')
    display(linked_tornadoes[['tornado_id', 'ef_rating', 'ncei_accepted_records',
                             'footprint_accepted_records', 'linked_county_years',
                             'linked_county_population_sum', 'suggested_split_group']].head())
    display(crosswalk.loc[crosswalk['plausible'] & ~crosswalk['accepted'],
                         ['source_table', 'source_id', 'tornado_id', 'evidence_tier',
                          'match_status', 'decision_reason', 'interval_outside_minutes',
                          'source_extent_distance_km']].head(10))
    print('Accepted links are automatic research decisions; review before scientific use.')

## SPC tracks and EF-label balance

SPC is the primary event catalog. These are historical single-track records, not preliminary daily reports. The default period is entirely within the EF era (which began February 1, 2007). The choice of 2010 is project scope, not a rating-scale transition or a DAT-completeness cutoff.


In [ ]:
spc_output = next((row for row in manifest.get('outputs', []) if row['source'] == 'SPC'), None)
spc = pd.DataFrame()
if spc_output:
    spc_path = local_path(DATA, spc_output['path'])
    if spc_path.is_file():
        spc = pd.read_csv(spc_path, dtype=str, keep_default_na=False)
if not spc.empty:
    print(f'{len(spc):,} SPC track records')
    columns = [c for c in ['yr','om','date','time','tz','st','mag','slat','slon','len','wid'] if c in spc]
    display(spc[columns].head(SAMPLE_ROWS))
    ratings = spc['mag'].where(spc['mag'].isin(list('012345')), 'Unknown')
    rating_counts = ratings.value_counts().reindex([*list('012345'), 'Unknown'], fill_value=0)
    display(rating_counts.rename_axis('EF rating').to_frame('tracks'))
    ax = rating_counts.plot.bar(figsize=(8, 3), color='#2878a8', title='SPC recorded EF ratings')
    ax.set_ylabel('Tracks'); ax.set_xlabel('EF rating'); ax.tick_params(axis='x', rotation=0)
    plt.tight_layout(); plt.show()
else:
    print('SPC data not available in the selected manifest yet.')


## Original-source yearly coverage and footprint quality

Annual outputs with zero records differ from unavailable outputs. Footprints can be nested damage regions; their counts and ratings are not independent tornado counts. The catalog fills some DAT gaps with Storm Events records but does not establish complete coverage. Source annual partitions can differ from UTC year at New Year.

In [ ]:
coverage_rows = []
for item in manifest.get('year_coverage', []):
    for year in range(START_YEAR, END_YEAR + 1):
        coverage_rows.append({'source': item['source'], 'product': item['product'], 'year': year,
                              'rows': item['rows_by_year'].get(str(year), 0),
                              'available': year in item['available_years']})
if coverage_rows:
    display(pd.DataFrame(coverage_rows).pivot(index='year', columns=['source','product'], values='rows'))
    unavailable = pd.DataFrame(coverage_rows).query('not available')
    if not unavailable.empty:
        display(unavailable)
else:
    print('No original-source coverage report yet.')
if quality:
    quality_ref = {'path': quality['download_manifest'], 'sha256': quality['download_manifest_sha256']}
    display(pd.DataFrame([reference_check(DATA, quality_ref)]))
    footprint_quality = quality.get('footprints_by_year', [])
    if footprint_quality:
        display(pd.DataFrame(footprint_quality))
else:
    print('No saved label/footprint-quality summary yet.')


## NCEI and footprint record samples

NCEI previews use tornado-only extracts. Footprint previews read the first available annual file. The catalog's `source` identifies DAT or SED (Storm Events). Its `event_id` is a name, not NCEI EVENT_ID; SED footprints have generated IDs. Neither preview performs event matching.

In [ ]:
for table in ('details', 'fatalities', 'locations'):
    output = next((row for row in manifest.get('outputs', [])
                   if row.get('source') == 'NCEI' and row.get('year') == NCEI_YEAR and row.get('table') == table), None)
    rows = csv_preview(local_path(DATA, output['path']), SAMPLE_ROWS) if output else []
    print(f'NCEI {NCEI_YEAR} {table}:')
    display(pd.DataFrame(rows)) if rows else print('No local sample available.')
rows = footprint_preview(DATA, manifest, SAMPLE_ROWS)
if rows:
    frame = pd.DataFrame(rows)
    display(frame[[c for c in ['objectid','source','stormdate','efscale','max_efscale','width','parents','children','geometry_type'] if c in frame]])
else:
    print('No local footprint sample available.')


## County population and housing context

One row per county/year, covering the 50 states and DC. These are July 1 estimates:
vintage 2020 supplies 2010–2019, and vintage 2025 supplies 2020–2025. They use each
release's county definitions, not necessarily those in effect on the tornado date.

Housing units are residences, not building counts. This table is not joined to
tornadoes or paths. A matching map FIPS code does not establish compatible boundaries;
Connecticut's newer planning regions are flagged as absent from the fixed 2020 map.
Do not fill missing joins with zero or interpret county totals as people directly hit.


In [ ]:
county_context = pd.DataFrame()
if census_manifest.get('status') == 'complete':
    context_path = local_path(DATA, census_manifest['context']['path'])
    if context_path.is_file():
        county_context = pd.read_csv(context_path, dtype={'county_fips': str, 'map_2020_fips_present': str})
if not county_context.empty:
    display(county_context.head(SAMPLE_ROWS))
    display(county_context.groupby(['year', 'estimate_vintage']).agg(
        counties=('county_fips', 'size'), population=('population', 'sum'),
        housing_units=('housing_units', 'sum')))
    unmatched = county_context.loc[county_context['map_2020_fips_present'].eq('false'),
                                   ['county_fips', 'state_name', 'county_name']].drop_duplicates()
    print(f'{len(unmatched)} county codes absent from the fixed 2020 map:')
    if not unmatched.empty:
        display(unmatched)
else:
    print('No completed Census context collection yet.')


## Simplified county map and tornado locations

The 2020 Census cartographic map is generalized at 1:5,000,000 and converted from
WGS84 KML to GeoJSON, preserving polygon parts and holes. It includes Puerto Rico and other US territories;
population/housing tables cover the 50 states and DC. The preview below shows the
contiguous US and SPC start points. It performs no spatial joins or exposure estimates.
Use the footprint table for available surveyed/reconstructed damage regions; connecting SPC endpoints
only approximates a track. Map boundaries are for display, not precise path intersections.


In [ ]:
from matplotlib.collections import LineCollection

county_map = {}
if census_manifest.get('status') == 'complete':
    county_map = load_json(local_path(DATA, census_manifest['boundaries']['path']), {})
if county_map.get('features'):
    rings = [ring for feature in county_map['features']
             for polygon in feature['geometry']['coordinates'] for ring in polygon]
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.add_collection(LineCollection(rings, colors='#aab3ba', linewidths=0.25))
    if not spc.empty and {'slon', 'slat'}.issubset(spc.columns):
        ax.scatter(pd.to_numeric(spc['slon'], errors='coerce'),
                   pd.to_numeric(spc['slat'], errors='coerce'),
                   s=2, alpha=0.18, color='#b84235', rasterized=True, label='SPC start locations')
        ax.legend(loc='lower left')
    ax.set(xlim=(-125, -66), ylim=(24, 50), xlabel='Longitude', ylabel='Latitude',
           title='Tornado start locations and simplified 2020 counties — contiguous US')
    ax.set_aspect(1.3)
    plt.tight_layout(); plt.show()
else:
    print('No completed county map collection yet.')


## Interpretation before analysis

- Use SPC as the primary tornado catalog; footprint availability should not define inclusion.
- Footprints are damage regions, sometimes nested; use the optional crosswalk evidence and review uncertain identities before modeling.
- Preserve raw footprint widths: `0.99` is a missing-width display placeholder and zero is not a usable path width. `path_width_yards` makes these values null.
- Footprint `ef_rating` describes a region; `max_ef_rating` is the catalog's maximum across associated regions. Neither replaces the SPC target without matching/review.
- The catalog does not provide individual DAT damage-indicator observations or original NCEI event IDs for SED records.
- Preserve unknown ratings; exclude them only when an experiment needs known supervised labels.
- Census population/housing are county context, not exact people/buildings hit. Reconcile geographic vintages before joins.
- Exclude rating-derived fields, wind estimates, and rating-revealing narratives from predictor sets.
- Group related records before train/test splits and assess rare EF classes carefully.

See [ANALYSIS.md](../docs/ANALYSIS.md) for column definitions and [the footprint audit](../reports/footprints/catalog_audit.md) for the source comparison. Reviewing uncertain links and selecting model predictors remain separate work; see [LINKAGE.md](../docs/LINKAGE.md).
